# Chapter20. Aggregation and Grouping

## 20.1 Planets Data

이 장에서는 Seaborn 라이브러리에 포함된 planets dataset을 사용합니다.

In [37]:
import numpy as np
import pandas as pd
import seaborn as sns

planets = sns.load_dataset('planets')
print(planets.shape)
print(planets.head())

(1035, 6)
            method  number  orbital_period   mass  distance  year
0  Radial Velocity       1         269.300   7.10     77.40  2006
1  Radial Velocity       1         874.774   2.21     56.95  2008
2  Radial Velocity       1         763.000   2.60     19.84  2011
3  Radial Velocity       1         326.030  19.40    110.62  2007
4  Radial Velocity       1         516.220  10.50    119.47  2009


💡  
* method: 발견 방법 (Radial Velocity, Transit 등)  
* number: 몇 개의 행성인지  
* orbital_period: 공전 주기 (일)  
* mass: 질량 (목성 질량 단위)  
* distance: 거리 (파섹)  
* year: 발견 년도

## 20.2 Simple Aggregation in Pandas

### Series aggregation

In [38]:
rng = np.random.RandomState(42)
ser = pd.Series(rng.rand(5))
print(ser)

0    0.374540
1    0.950714
2    0.731994
3    0.598658
4    0.156019
dtype: float64


In [39]:
print("Sum:", ser.sum())
print("Average:", ser.mean())
print("Min:", ser.min())
print("Max:", ser.max())

Sum: 2.811925491708157
Average: 0.5623850983416314
Min: 0.15601864044243652
Max: 0.9507143064099162


### DataFrame aggregation (렬 기준)

In [40]:
df = pd.DataFrame({'A': rng.rand(5), 'B': rng.rand(5)})
print(df)

          A         B
0  0.155995  0.020584
1  0.058084  0.969910
2  0.866176  0.832443
3  0.601115  0.212339
4  0.708073  0.181825


In [41]:
print("Each column average:")
print(df.mean())

Each column average:
A    0.477888
B    0.443420
dtype: float64


In [42]:
print("Each row average:")
print(df.mean(axis='columns'))

Each row average:
0    0.088290
1    0.513997
2    0.849309
3    0.406727
4    0.444949
dtype: float64


### describe() method

describe()는 여러 집계 결과를 한 번에 보여줍니다.

In [43]:
print(planets.dropna().describe())

          number  orbital_period        mass    distance         year
count  498.00000      498.000000  498.000000  498.000000   498.000000
mean     1.73494      835.778671    2.509320   52.068213  2007.377510
std      1.17572     1469.128259    3.636274   46.596041     4.167284
min      1.00000        1.328300    0.003600    1.350000  1989.000000
25%      1.00000       38.272250    0.212500   24.497500  2005.000000
50%      1.00000      357.000000    1.245000   39.940000  2009.000000
75%      2.00000      999.600000    2.867500   59.332500  2011.000000
max      6.00000    17337.500000   25.000000  354.000000  2014.000000


## 20.3 groupby: Split, Apply, Combine

### GroupBy의 핵심 개념: Split-Apply-Combine

1. Split (분할)  : 데이터를 특정 기준(Key)으로 나눈다
2. Apply (적용)  : 각 그룹에 함수를 적용한다 (집계, 변환, 필터링 등)
3. Combine (결합): 결과를 다시 합친다

### Example

In [44]:
df = pd.DataFrame({'key': ['A', 'B', 'C', 'A', 'B', 'C'], 
                   'data': range(6)})
print(df)

  key  data
0   A     0
1   B     1
2   C     2
3   A     3
4   B     4
5   C     5


In [45]:
# 'key'별로 group화하여 합계 계산
print(df.groupby('key').sum())

     data
key      
A       3
B       5
C       7


💡 df.groupby('key')는 GroupBy 객체를 반환하며, 이 객체에 .sum() 같은 집계 함수를 적용해야 실제 계산이 수행됩니다.

## 20.4 The GroupBy Object

### 20.4.1 Column indexing

In [46]:
print(planets.groupby('method')['orbital_period'].median())

method
Astrometry                         631.180000
Eclipse Timing Variations         4343.500000
Imaging                          27500.000000
Microlensing                      3300.000000
Orbital Brightness Modulation        0.342887
Pulsar Timing                       66.541900
Pulsation Timing Variations       1170.000000
Radial Velocity                    360.200000
Transit                              5.714932
Transit Timing Variations           57.011000
Name: orbital_period, dtype: float64


### 20.4.2 Group iteration

In [47]:
for (method, group) in planets.groupby('method'):
    print(f"{method:30s} shape={group.shape}")

Astrometry                     shape=(2, 6)
Eclipse Timing Variations      shape=(9, 6)
Imaging                        shape=(38, 6)
Microlensing                   shape=(23, 6)
Orbital Brightness Modulation  shape=(3, 6)
Pulsar Timing                  shape=(5, 6)
Pulsation Timing Variations    shape=(1, 6)
Radial Velocity                shape=(553, 6)
Transit                        shape=(397, 6)
Transit Timing Variations      shape=(4, 6)


### 20.4.3 Dispatch Methods

GroupBy 객체는 describe() 같은 method를 각 group에 적용합니다.

In [48]:
print(planets.groupby('method')['year'].describe().unstack())

       method                       
count  Astrometry                          2.0
       Eclipse Timing Variations           9.0
       Imaging                            38.0
       Microlensing                       23.0
       Orbital Brightness Modulation       3.0
                                         ...  
max    Pulsar Timing                    2011.0
       Pulsation Timing Variations      2007.0
       Radial Velocity                  2014.0
       Transit                          2014.0
       Transit Timing Variations        2014.0
Length: 80, dtype: float64


## 20.5 Aggregate, Filter, Transform, Apply

### Data for example

In [53]:
rng = np.random.RandomState(0)
df = pd.DataFrame({'key': ['A', 'B', 'C', 'A', 'B', 'C'],
                   'data1': range(6),
                   'data2': rng.randint(0, 10, 6)},
                   columns=['key', 'data1', 'data2'])
print(df)

  key  data1  data2
0   A      0      5
1   B      1      0
2   C      2      3
3   A      3      3
4   B      4      7
5   C      5      9


### 20.5.1 Aggregate

In [54]:
print(df.groupby('key').aggregate(['min', np.median, 'max']))

    data1            data2           
      min median max   min median max
key                                  
A       0    1.5   3     3    4.0   5
B       1    2.5   4     0    3.5   7
C       2    3.5   5     3    6.0   9


In [55]:
print(df.groupby('key').aggregate({'data1': 'min', 'data2': 'max'}))

     data1  data2
key              
A        0      5
B        1      7
C        2      9


### 20.5.2 Filtering

Group의 특성에 따라 Group 전체를 포함하거나 제외합니다.

In [56]:
def filter_func(x):
    return x['data2'].std() > 4

print(df.groupby('key').filter(filter_func))

  key  data1  data2
1   B      1      0
2   C      2      3
4   B      4      7
5   C      5      9


💡 Group A는 data2의 표준편차가 4 이하이므로 제외되었습니다.

### 20.5.3 Transform

Group별로 변환을 적용하며, 출력 형태는 입력과 동일합니다.

In [57]:
def center(x):
    return x - x.mean()

print(df.groupby('key').transform(center))

   data1  data2
0   -1.5    1.0
1   -1.5   -3.5
2   -1.5   -3.0
3    1.5   -1.0
4    1.5    3.5
5    1.5    3.0


### 20.5.4 The Apply method

가장 유연한 방법으로, 사용자 정의 함수를 Group에 적용할 수 있습니다.

In [58]:
def norm_by_data(x):
    # x is a DataFrame of group values
    x['data1'] = x['data1'] / x['data2'].sum()
    return x

print(df.groupby('key').apply(norm_by_data))

          data1  data2
key                   
A   0  0.000000      5
    3  0.375000      3
B   1  0.142857      0
    4  0.571429      7
C   2  0.166667      3
    5  0.416667      9


## 20.6 Specifying the Split Key

### A list, array, series, or index providing the grouping keys

In [60]:
L = [0, 1, 0, 1, 2, 0]
print(df[['data1', 'data2']].groupby(L).sum())

   data1  data2
0      7     17
1      4      3
2      4      7


### A dictionary or series mapping index to group

In [61]:
df2 = df.set_index('key')
mapping = {'A': 'vowel', 'B': 'consonant', 'C': 'consonant'}
print(df2.groupby(mapping).sum())

           data1  data2
key                    
consonant     12     19
vowel          3      8


### Any Python function

In [67]:
print(df2.groupby(str.lower).mean())

     data1  data2
key              
a      1.5    4.0
b      2.5    3.5
c      3.5    6.0


## 20.7 Grouping Example

발견 방법(method)과 년대(decade)별로 group화하여 외계행성 발견 개수 계산

In [71]:
decade = 10 * (planets['year'] // 10)
decade = decade.astype(str) + 's'
decade.name = 'decade'

result = planets.groupby(['method', decade])['number'].sum().unstack().fillna(0)
print(result)

decade                         1980s  1990s  2000s  2010s
method                                                   
Astrometry                       0.0    0.0    0.0    2.0
Eclipse Timing Variations        0.0    0.0    5.0   10.0
Imaging                          0.0    0.0   29.0   21.0
Microlensing                     0.0    0.0   12.0   15.0
Orbital Brightness Modulation    0.0    0.0    0.0    5.0
Pulsar Timing                    0.0    9.0    1.0    1.0
Pulsation Timing Variations      0.0    0.0    1.0    0.0
Radial Velocity                  1.0   52.0  475.0  424.0
Transit                          0.0    0.0   64.0  712.0
Transit Timing Variations        0.0    0.0    0.0    9.0
